<a href="https://colab.research.google.com/github/routparam12/Python_qns/blob/main/model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Create a model for prediction

In [25]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import classification_report, accuracy_score
import numpy as np

In [26]:
df = pd.read_csv('/content/insurance.csv')

In [27]:
df.sample(5)

,age,weight,height,income_lpa,smoker,city,occupation,insurance_premium_category
30,35,89.6,1.73,32.97,False,Delhi,business_owner,Low
8,73,58.0,1.58,1.78,False,Chandigarh,retired,Medium
79,39,55.3,1.56,30.00,True,Indore,government_job,Low
35,59,59.3,1.69,43.28,True,Chandigarh,private_job,Medium
84,75,86.2,1.73,0.62,True,Jaipur,retired,High


In [28]:
df['occupation'].unique()

array(['retired', 'freelancer', 'student', 'government_job',
       'business_owner', 'unemployed', 'private_job'], dtype=object)

In [29]:
df_feat = df.copy()

In [30]:
#Feature 1: BMI
df_feat["bmi"]= df_feat["weight"]/(df_feat["height"]/100)**2

In [31]:
# Feature 2 : Age Group
def age_group(age):
  if age < 25:
    return "Young"
  elif age < 45:
    return "Adult"
  elif age < 65:
    return "Middle Age"
  else:
    return "Senior"

In [32]:
df_feat["age_group"] = df_feat["age"].apply(age_group)

In [33]:
#Feature 3: Lifestyle Risk
def lifestyle_risk(row):
  if row["smoker"] and row["bmi"] > 30:
    return 'high'
  elif row["smoker"] or row["bmi"] > 27:
    return 'medium'
  else :
    return "low"

In [34]:
df_feat["lifestyle_risk"] = df_feat.apply(lifestyle_risk, axis=1)

In [35]:
tier_1_cities = ["Mumbai", "Delhi", "Bangalore", "Chennai", "Kolkata", "Hyderabad", "Pune"]
tier_2_cities = [
    "Jaipur", "Chandigarh", "Indore", "Lucknow", "Patna", "Ranchi", "Visakhapatnam", "Coimbatore",
    "Bhopal", "Nagpur", "Vadodara", "Surat", "Rajkot", "Jodhpur", "Raipur", "Amritsar", "Varanasi",
    "Agra", "Dehradun", "Mysore", "Jabalpur", "Guwahati", "Thiruvananthapuram", "Ludhiana", "Nashik",
    "Allahabad", "Udaipur", "Aurangabad", "Hubli", "Belgaum", "Salem", "Vijayawada", "Tiruchirappalli",
    "Bhavnagar", "Gwalior", "Dhanbad", "Bareilly", "Aligarh", "Gaya", "Kozhikode", "Warangal",
    "Kolhapur", "Bilaspur", "Jalandhar", "Noida", "Guntur", "Asansol", "Siliguri"
]

In [36]:
#Feature 4: city Tier
def city_tier(city):
  if city in tier_1_cities:
    return 1
  elif city in tier_2_cities:
    return 2
  else:
    return 3


In [37]:
df_feat["city_tier"] = df_feat["city"].apply(city_tier)

In [38]:
df_feat.drop(columns=['age', 'weight', 'height', 'smoker', 'city'])[['income_lpa', 'occupation', 'bmi', 'age_group', 'lifestyle_risk', 'city_tier', 'insurance_premium_category']].sample(5)


,income_lpa,occupation,bmi,age_group,lifestyle_risk,city_tier,insurance_premium_category
32,50.00,private_job,314958.448753,Middle Age,medium,2,Medium
78,14.74,freelancer,279327.976625,Middle Age,high,2,High
39,11.99,unemployed,356434.240363,Middle Age,high,1,High
76,1.12,retired,440444.444444,Middle Age,medium,2,High
73,2.22,retired,321216.278006,Senior,high,1,High


In [39]:
# Select features and target
X = df_feat[["bmi", "age_group", "lifestyle_risk", "city_tier", "income_lpa", "occupation"]]
y = df_feat["insurance_premium_category"]

In [40]:
X

,bmi,age_group,lifestyle_risk,city_tier,income_lpa,occupation
0,492274.819198,Senior,medium,2,2.92000,retired
1,301890.172893,Adult,medium,1,34.28000,freelancer
2,211183.819155,Adult,medium,2,36.64000,freelancer
3,455359.001041,Young,high,1,3.34000,student
4,242968.750000,Senior,high,2,3.94000,retired
...,...,...,...,...,...,...
95,214207.472920,Adult,medium,2,19.64000,business_owner
96,479844.830494,Adult,medium,1,34.01000,private_job
97,187654.320988,Middle Age,medium,1,44.86000,freelancer
98,305216.761261,Adult,medium,1,28.30000,business_owner


In [41]:
y

,insurance_premium_category
0,High
1,Low
2,Low
3,Medium
4,High
...,...
95,Low
96,Low
97,Low
98,Low


In [42]:
# Define categorical and numeric features
categorical_features = ["age_group", "lifestyle_risk", "occupation", "city_tier"]
numeric_features = ["bmi", "income_lpa"]

In [43]:
# Create column transformer for OHE
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(), categorical_features),
        ("num", "passthrough", numeric_features)
    ]
)


# Create a pipeline with preprocessing and random forest classifier
pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(random_state=42))
])


# Split data and train model
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)
pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat', OneHotEncoder(),
                                                  ['age_group',
                                                   'lifestyle_risk',
                                                   'occupation', 'city_tier']),
                                                 ('num', 'passthrough',
                                                  ['bmi', 'income_lpa'])])),
                ('classifier', RandomForestClassifier(random_state=42))])

In [44]:
# Predict and evaluate
y_pred = pipeline.predict(X_test)
accuracy_score(y_test, y_pred)


0.85

In [45]:
X_test.sample(5)


,bmi,age_group,lifestyle_risk,city_tier,income_lpa,occupation
52,473447.198669,Young,medium,2,2.96,student
92,183199.415632,Adult,high,2,30.00,government_job
82,178748.124741,Adult,medium,1,12.96,unemployed
44,300781.250000,Middle Age,high,2,50.00,private_job
51,388279.225728,Middle Age,high,2,28.95,private_job


In [46]:

import pickle

# Save the trained pipeline using pickle
pickle_model_path = "model.pkl"
with open(pickle_model_path, "wb") as f:
    pickle.dump(pipeline, f)
